# Bitcoin Mempool Transaction Analysis Dashboard
Comprehensive visualization of mempool transaction data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)
%matplotlib inline

In [2]:
# Read the CSV file
df = pd.read_csv('mempool_log.csv')

# Filter out placeholder/test data
df = df[df['txid'] != 'your_txid_here'].copy()

# Convert time to datetime
df['time'] = pd.to_datetime(df['time'])

# Calculate fee rate (satoshis per vbyte)
df['fee_rate'] = df['fee'] / df['vsize']

# Count number of ancestors
df['ancestor_count'] = df['ancestors'].apply(lambda x: len(str(x).split(';')) if pd.notna(x) and str(x) != '' else 0)

print(f"Loaded {len(df)} transactions")
df.head()

ParserError: Error tokenizing data. C error: Expected 5 fields in line 62, saw 9


## 1. Transaction Timeline Analysis

In [ ]:
# Transaction Timeline
plt.figure(figsize=(18, 6))
scatter = plt.scatter(df['time'], df['fee_rate'], c=df['vsize'], cmap='viridis', alpha=0.6, s=100, edgecolors='black', linewidth=0.5)
plt.xlabel('Time', fontsize=13, fontweight='bold')
plt.ylabel('Fee Rate (sat/vB)', fontsize=13, fontweight='bold')
plt.title('Transaction Timeline: Fee Rate vs Time (sized by vsize)', fontsize=15, fontweight='bold', pad=20)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
cbar = plt.colorbar(scatter)
cbar.set_label('Virtual Size (vB)', fontsize=11)
plt.tight_layout()
plt.show()

## 2. Fee Rate Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fee Rate Distribution
axes[0].hist(df['fee_rate'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Fee Rate (sat/vB)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Fee Rate Distribution', fontsize=12, fontweight='bold')
axes[0].axvline(df['fee_rate'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["fee_rate"].median():.2f}')
axes[0].axvline(df['fee_rate'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {df["fee_rate"].mean():.2f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Virtual Size Distribution
axes[1].hist(df['vsize'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Virtual Size (vB)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Transaction Size Distribution', fontsize=12, fontweight='bold')
axes[1].axvline(df['vsize'].median(), color='blue', linestyle='--', linewidth=2, label=f'Median: {df["vsize"].median():.0f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Total Fee Distribution
axes[2].hist(df['fee'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Total Fee (satoshis)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[2].set_title('Total Fee Distribution', fontsize=12, fontweight='bold')
axes[2].axvline(df['fee'].median(), color='purple', linestyle='--', linewidth=2, label=f'Median: {df["fee"].median():.0f}')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Fee vs Size Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fee vs Size Scatter
scatter = axes[0].scatter(df['vsize'], df['fee'], c=df['fee_rate'], cmap='plasma', alpha=0.6, s=80, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Virtual Size (vB)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Fee (satoshis)', fontsize=12, fontweight='bold')
axes[0].set_title('Fee vs Transaction Size', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Fee Rate (sat/vB)', fontsize=10)

# Fee Rate vs Size
axes[1].scatter(df['vsize'], df['fee_rate'], alpha=0.5, s=60, color='#E63946', edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Virtual Size (vB)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fee Rate (sat/vB)', fontsize=12, fontweight='bold')
axes[1].set_title('Fee Rate vs Transaction Size', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Ancestor Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ancestor Count Distribution
ancestor_counts = df['ancestor_count'].value_counts().sort_index()
axes[0].bar(ancestor_counts.index, ancestor_counts.values, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Ancestors', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Transaction Count', fontsize=12, fontweight='bold')
axes[0].set_title('Ancestor Count Distribution', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Fee Rate vs Ancestor Count
axes[1].scatter(df['ancestor_count'], df['fee_rate'], alpha=0.5, s=60, color='#457B9D', edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Number of Ancestors', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fee Rate (sat/vB)', fontsize=12, fontweight='bold')
axes[1].set_title('Fee Rate vs Ancestor Count', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Top Transactions Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 10 by Fee
top_fees = df.nlargest(10, 'fee')[['fee', 'fee_rate', 'vsize']].copy()
top_fees['label'] = [f'TX{i+1}' for i in range(len(top_fees))]
bars = axes[0, 0].bar(range(len(top_fees)), top_fees['fee'], color='#E63946', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Transaction', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Fee (satoshis)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Top 10 Transactions by Total Fee', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(range(len(top_fees)))
axes[0, 0].set_xticklabels(top_fees['label'])
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, (bar, fee) in enumerate(zip(bars, top_fees['fee'])):
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height, f'{int(fee):,}', ha='center', va='bottom', fontsize=8)

# Top 10 by Fee Rate
top_fee_rates = df.nlargest(10, 'fee_rate')[['fee', 'fee_rate', 'vsize']].copy()
top_fee_rates['label'] = [f'TX{i+1}' for i in range(len(top_fee_rates))]
bars = axes[0, 1].bar(range(len(top_fee_rates)), top_fee_rates['fee_rate'], color='#F4A261', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Transaction', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Fee Rate (sat/vB)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Top 10 Transactions by Fee Rate', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(range(len(top_fee_rates)))
axes[0, 1].set_xticklabels(top_fee_rates['label'])
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, (bar, rate) in enumerate(zip(bars, top_fee_rates['fee_rate'])):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height, f'{rate:.1f}', ha='center', va='bottom', fontsize=8)

# Top 10 by Size
top_sizes = df.nlargest(10, 'vsize')[['fee', 'fee_rate', 'vsize']].copy()
top_sizes['label'] = [f'TX{i+1}' for i in range(len(top_sizes))]
bars = axes[1, 0].bar(range(len(top_sizes)), top_sizes['vsize'], color='#2A9D8F', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Transaction', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Virtual Size (vB)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Top 10 Transactions by Size', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(range(len(top_sizes)))
axes[1, 0].set_xticklabels(top_sizes['label'])
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, (bar, size) in enumerate(zip(bars, top_sizes['vsize'])):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height, f'{int(size)}', ha='center', va='bottom', fontsize=8)

# Cumulative Fee Over Time
df_sorted = df.sort_values('time')
axes[1, 1].plot(df_sorted['time'], df_sorted['fee'].cumsum(), linewidth=2, color='#264653')
axes[1, 1].fill_between(df_sorted['time'], df_sorted['fee'].cumsum(), alpha=0.3, color='#264653')
axes[1, 1].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Cumulative Fees (satoshis)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Cumulative Transaction Fees Over Time', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 6. Time-based Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Transaction Rate Over Time
df_sorted = df.sort_values('time')
df_sorted['minute'] = df_sorted['time'].dt.floor('T')
tx_per_minute = df_sorted.groupby('minute').size()
axes[0].plot(tx_per_minute.index, tx_per_minute.values, marker='o', linewidth=2, markersize=6, color='#06A77D')
axes[0].fill_between(tx_per_minute.index, tx_per_minute.values, alpha=0.3, color='#06A77D')
axes[0].set_xlabel('Time', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Transactions per Minute', fontsize=12, fontweight='bold')
axes[0].set_title('Transaction Rate Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Average Fee Rate Over Time
avg_fee_rate = df_sorted.groupby('minute')['fee_rate'].mean()
axes[1].plot(avg_fee_rate.index, avg_fee_rate.values, marker='s', linewidth=2, markersize=6, color='#E76F51')
axes[1].fill_between(avg_fee_rate.index, avg_fee_rate.values, alpha=0.3, color='#E76F51')
axes[1].set_xlabel('Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Fee Rate (sat/vB)', fontsize=12, fontweight='bold')
axes[1].set_title('Average Fee Rate Over Time', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print("="*70)
print("BITCOIN MEMPOOL TRANSACTION ANALYSIS SUMMARY")
print("="*70)
print(f"\nTotal Transactions Analyzed: {len(df)}")
print(f"Time Range: {df['time'].min()} to {df['time'].max()}")
print(f"Duration: {(df['time'].max() - df['time'].min()).total_seconds() / 60:.1f} minutes")

print(f"\n{'FEE STATISTICS':^70}")
print("-"*70)
print(f"  Average Fee Rate:        {df['fee_rate'].mean():>12.2f} sat/vB")
print(f"  Median Fee Rate:         {df['fee_rate'].median():>12.2f} sat/vB")
print(f"  Min Fee Rate:            {df['fee_rate'].min():>12.2f} sat/vB")
print(f"  Max Fee Rate:            {df['fee_rate'].max():>12.2f} sat/vB")
print(f"  Std Dev Fee Rate:        {df['fee_rate'].std():>12.2f} sat/vB")

print(f"\n{'TRANSACTION SIZE STATISTICS':^70}")
print("-"*70)
print(f"  Average Size:            {df['vsize'].mean():>12.0f} vB")
print(f"  Median Size:             {df['vsize'].median():>12.0f} vB")
print(f"  Min Size:                {df['vsize'].min():>12.0f} vB")
print(f"  Max Size:                {df['vsize'].max():>12.0f} vB")
print(f"  Total Volume:            {df['vsize'].sum():>12,} vB")

print(f"\n{'TOTAL FEE STATISTICS':^70}")
print("-"*70)
print(f"  Total Fees Collected:    {df['fee'].sum():>12,} satoshis")
print(f"  Average Fee:             {df['fee'].mean():>12,.0f} satoshis")
print(f"  Median Fee:              {df['fee'].median():>12,.0f} satoshis")
print(f"  Min Fee:                 {df['fee'].min():>12,} satoshis")
print(f"  Max Fee:                 {df['fee'].max():>12,} satoshis")

print(f"\n{'ANCESTOR STATISTICS':^70}")
print("-"*70)
print(f"  Average Ancestor Count:  {df['ancestor_count'].mean():>12.1f}")
print(f"  Median Ancestor Count:   {df['ancestor_count'].median():>12.0f}")
print(f"  Max Ancestor Count:      {df['ancestor_count'].max():>12.0f}")
print(f"  Transactions w/ Ancestors: {(df['ancestor_count'] > 0).sum():>10}")

print("\n" + "="*70)

# Display detailed statistics table
print("\nDetailed Statistics Table:")
stats_df = df[['fee', 'fee_rate', 'vsize', 'ancestor_count']].describe()
print(stats_df)

## 8. Correlation Heatmap

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df[['fee', 'vsize', 'fee_rate', 'ancestor_count']].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Transaction Metrics', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 9. Category Analysis

In [ ]:
# Create categories
df['size_category'] = pd.cut(df['vsize'], bins=[0, 150, 300, 500, 1000, 10000], 
                              labels=['Tiny (<150)', 'Small (150-300)', 'Medium (300-500)', 
                                     'Large (500-1K)', 'XLarge (>1K)'])
df['fee_category'] = pd.cut(df['fee_rate'], bins=[0, 100, 200, 500, 1000, 100000],
                             labels=['Low', 'Medium', 'High', 'Very High', 'Extreme'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Size category distribution
size_counts = df['size_category'].value_counts()
axes[0].pie(size_counts.values, labels=size_counts.index, autopct='%1.1f%%', startangle=90, 
            colors=sns.color_palette('Set2', len(size_counts)))
axes[0].set_title('Transaction Distribution by Size Category', fontsize=13, fontweight='bold')

# Fee category distribution
fee_counts = df['fee_category'].value_counts()
axes[1].pie(fee_counts.values, labels=fee_counts.index, autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('YlOrRd', len(fee_counts)))
axes[1].set_title('Transaction Distribution by Fee Rate Category', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Heatmap of size vs fee categories
plt.figure(figsize=(10, 6))
heatmap_data = pd.crosstab(df['size_category'], df['fee_category'])
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd', 
            cbar_kws={'label': 'Transaction Count'})
plt.xlabel('Fee Rate Category', fontsize=12, fontweight='bold')
plt.ylabel('Size Category', fontsize=12, fontweight='bold')
plt.title('Transaction Distribution: Size vs Fee Rate Category', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()